# Dockerfile & Docker Compose for API and Postgres 🐳

**Week 4 · Role 3 (Member 3 — Wednesday)**

Every backend team hits the same wall sooner or later: *"The API runs fine on my laptop, why is it crashing on yours?"* One person has Python 3.11, another has Python 3.13, and someone else does not have PostgreSQL running on port `5432`.

As **Member 3**, your job is to solve that problem for the whole team. By the end of this notebook, everyone on the team will clearly understand:

1. **Image vs. Container** — the exact difference between a frozen blueprint and a live running box, and why data disappears unless we attach a volume.
2. **A Modern `Dockerfile`** — how to package a FastAPI + `uv` service so rebuilds stay fast and secure.
3. **`compose.yaml` with Health Checks** — how to wire the **API** and **PostgreSQL** together on a private network, prevent the "database not ready yet" startup crash, and verify the stack runs cleanly.

> **How to use this notebook:** Press **Shift + Enter** on each code cell from top to bottom. Every code cell runs with standard Python so you can explore simulations, generate a complete reference stack, and audit the files automatically.

## 1. Image vs. Container: The Blueprint and the Running Kitchen

People often mix up the words **image** and **container**. Here is the simplest way to picture them:

| Concept | Everyday Analogy | Programming Analogy | Can it change while running? | How many can exist? |
|---|---|---|---|---|
| **Docker Image** | A laminated recipe card | A Python `class` | **No** (Read-only layers) | Built once, shared everywhere |
| **Docker Container** | A dish actively cooking on a stove | An `object` instance of that class | **Yes** (Has one thin scratchpad layer on top) | Spin up 1, 3, or 50 from the same image |

### What happens inside?
- An **image** is a stack of **read-only layers** (base Linux + Python + installed packages + your API code). Once built, an image never changes.
- When you start a **container**, Docker places **one thin writable layer** on top of the image and starts your process in an isolated room.
- **Crucial catch:** When you delete a container, that thin writable layer goes in the trash with it! That is awesome for a stateless FastAPI server, right up until you run **PostgreSQL**—which is why databases attach a **Named Volume** (a safe external hard drive managed by Docker) so rows survive container restarts and replacements.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class DockerImage:
    '''An image is frozen (immutable): its layers cannot be edited in place.'''
    tag: str
    layers: tuple[str, ...]

@dataclass
class Container:
    '''A container shares the frozen image and adds its own isolated scratch layer.'''
    name: str
    image: DockerImage
    volume: dict[str, str] | None = None

    def write_temp(self, path: str, content: str) -> None:
        self.scratch_layer[path] = content

    def write_volume(self, key: str, content: str) -> None:
        if self.volume is None:
            raise RuntimeError("No volume mounted!")
        self.volume[key] = content


# 1. Build ONE read-only image
api_image = DockerImage(
    tag="team-api:v1",
    layers=("debian-slim", "python-3.13 + uv", "fastapi + psycopg", "app/main.py"),
)

# 2. Start TWO independent containers from that SAME image
api_one = Container(name="api-worker-1", image=api_image)
api_two = Container(name="api-worker-2", image=api_image)

api_one.write_temp("/tmp/request.log", "GET /health -> 200")
print("Image layers (unchanged):", api_image.layers)
print("api-worker-1 scratchpad :", api_one.scratch_layer)
print("api-worker-2 scratchpad :", api_two.scratch_layer, " <-- completely isolated!\n")

# 3. Why PostgreSQL needs a Named Volume
postgres_data_volume: dict[str, str] = {}
pg_image = DockerImage(tag="postgres:16-alpine", layers=("alpine", "postgresql-16"))

db_container_old = Container(name="db-monday", image=pg_image, volume=postgres_data_volume)
db_container_old.write_temp("/tmp/cache.tmp", "lost when container is removed")
db_container_old.write_volume("users_row_1", "Ada Lovelace")

# Suppose we destroy Monday's container and create a brand-new container on Wednesday:
del db_container_old
db_container_new = Container(name="db-wednesday", image=pg_image, volume=postgres_data_volume)

print("New DB container scratchpad:", db_container_new.scratch_layer)
print("New DB container volume    :", db_container_new.volume, " <-- saved rows survived!")

Image layers (unchanged): ('debian-slim', 'python-3.13 + uv', 'fastapi + psycopg', 'app/main.py')
api-worker-1 scratchpad : {'/tmp/request.log': 'GET /health -> 200'}
api-worker-2 scratchpad : {}  <-- completely isolated!

New DB container scratchpad: {}
New DB container volume    : {'users_row_1': 'Ada Lovelace'}  <-- saved rows survived!


## 2. Building a Modern `Dockerfile` for FastAPI + `uv`

A `Dockerfile` is simply a top-to-bottom script that tells Docker how to bake an **image**. Every instruction (`FROM`, `COPY`, `RUN`) creates a cached layer. To keep our modern Python stack **fast, small, and safe**, we follow four golden rules:

1. **Keep junk and secrets out with `.dockerignore`:** Never copy `.env` (real API keys/passwords), `.git`, or your laptop's `.venv` folder into a Linux image.
2. **Copy `pyproject.toml` and `uv.lock` *before* copying `app/`:** Docker caches layers top-to-bottom. If a layer changes, every step *below* it runs again. Copying dependency files first means editing Python code in `app/` will **not** re-download packages!
3. **Run as a non-root user (`appuser`):** By default, containers run as `root`. Switching to an unprivileged user reduces damage if a bug is ever exploited.
4. **Bind Uvicorn to `0.0.0.0`, NOT `127.0.0.1`:**
   - On your laptop, `127.0.0.1` means *"only this laptop"*.
   - Inside a container, `127.0.0.1` means *"only inside this single container"*—neither your browser nor another container could reach the API! Binding `0.0.0.0` tells Uvicorn to listen on the container's network card so Docker can route traffic in.

In [3]:
import tempfile
from pathlib import Path

# Create a clean reference stack folder we will inspect and audit throughout the lesson
stack_dir = Path(tempfile.mkdtemp(prefix="week4_role3_stack_"))
(stack_dir / "app").mkdir(parents=True, exist_ok=True)

dockerignore_text = '''# Never bake secrets, git history, or host virtual environments into an image
.git
.venv
__pycache__/
*.py[cod]
.pytest_cache/
.ruff_cache/
.env
.env.*
!.env.example
tests/
docs/
'''

dockerfile_text = r'''FROM python:3.13-slim

# Grab the official uv binary for fast, deterministic dependency installs
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/

# Stream logs immediately (no buffering) and pre-compile bytecode for quick startup
ENV PYTHONUNBUFFERED=1 \
    UV_COMPILE_BYTECODE=1 \
    UV_LINK_MODE=copy

WORKDIR /app

# Step 1: Install dependencies FIRST so this heavy layer stays cached when app/ changes
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev --no-install-project

# Step 2: Create a non-root Linux user
RUN useradd --create-home --shell /bin/bash appuser

# Step 3: Copy application source code LAST
COPY --chown=appuser:appuser app ./app

USER appuser
EXPOSE 8000

# Bind to 0.0.0.0 so requests from outside the container reach FastAPI
CMD ["uv", "run", "--no-sync", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

(stack_dir / ".dockerignore").write_text(dockerignore_text, encoding="utf-8")
(stack_dir / "Dockerfile").write_text(dockerfile_text, encoding="utf-8")

print(f"Created .dockerignore and Dockerfile in: {stack_dir}\n")
for line_no, line in enumerate(dockerfile_text.splitlines(), start=1):
    print(f"{line_no:02d} | {line}")

Created .dockerignore and Dockerfile in: /tmp/week4_role3_stack_17keo8uw

01 | FROM python:3.13-slim
02 | 
03 | # Grab the official uv binary for fast, deterministic dependency installs
04 | COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/
05 | 
06 | # Stream logs immediately (no buffering) and pre-compile bytecode for quick startup
07 | ENV PYTHONUNBUFFERED=1 \
08 |     UV_COMPILE_BYTECODE=1 \
09 |     UV_LINK_MODE=copy
10 | 
11 | WORKDIR /app
12 | 
13 | # Step 1: Install dependencies FIRST so this heavy layer stays cached when app/ changes
14 | COPY pyproject.toml uv.lock ./
15 | RUN uv sync --frozen --no-dev --no-install-project
16 | 
17 | # Step 2: Create a non-root Linux user
18 | RUN useradd --create-home --shell /bin/bash appuser
19 | 
20 | # Step 3: Copy application source code LAST
21 | COPY --chown=appuser:appuser app ./app
22 | 
23 | USER appuser
24 | EXPOSE 8000
25 | 
26 | # Bind to 0.0.0.0 so requests from outside the container reach FastAPI
27 | CMD ["uv", "run", "-

In [4]:
# Why does instruction order in a Dockerfile save minutes every day? Let's test both orders!
def simulate_rebuild(steps: list[tuple[str, float, bool]], label: str) -> None:
    print(f"=== {label} ===")
    cache_broken = False
    total_seconds = 0.0
    for instruction, build_seconds, file_changed in steps:
        if file_changed:
            cache_broken = True
        status = "RE-RUN 🔄" if cache_broken else "CACHED ⚡"
        cost = build_seconds if cache_broken else 0.0
        total_seconds += cost
        print(f"  [{status}] {instruction:<46} (+{cost:4.1f}s)")
    print(f"  Total rebuild time after editing 1 line in app/main.py: {total_seconds:.1f}s\n")


# Scenario: A teammate edits ONE line in app/main.py, while pyproject.toml & uv.lock did NOT change.
bad_order = [
    ("FROM python:3.13-slim", 0.5, False),
    ("COPY . .  (copies app/main.py too early!)", 0.4, True),
    ("RUN uv sync --frozen --no-dev", 18.0, False),
]

good_order = [
    ("FROM python:3.13-slim", 0.5, False),
    ("COPY pyproject.toml uv.lock ./", 0.2, False),
    ("RUN uv sync --frozen --no-dev", 18.0, False),
    ("COPY app ./app  (only this tiny step re-runs!)", 0.3, True),
]

simulate_rebuild(bad_order, "Anti-Pattern: Copying everything before installing packages")
simulate_rebuild(good_order, "Our Dockerfile: Copying lockfiles first, code last")

=== Anti-Pattern: Copying everything before installing packages ===
  [CACHED ⚡] FROM python:3.13-slim                          (+ 0.0s)
  [RE-RUN 🔄] COPY . .  (copies app/main.py too early!)      (+ 0.4s)
  [RE-RUN 🔄] RUN uv sync --frozen --no-dev                  (+18.0s)
  Total rebuild time after editing 1 line in app/main.py: 18.4s

=== Our Dockerfile: Copying lockfiles first, code last ===
  [CACHED ⚡] FROM python:3.13-slim                          (+ 0.0s)
  [CACHED ⚡] COPY pyproject.toml uv.lock ./                 (+ 0.0s)
  [CACHED ⚡] RUN uv sync --frozen --no-dev                  (+ 0.0s)
  [RE-RUN 🔄] COPY app ./app  (only this tiny step re-runs!) (+ 0.3s)
  Total rebuild time after editing 1 line in app/main.py: 0.3s



## 3. Writing a Real `/health` Endpoint for API + Postgres

To verify that the API and PostgreSQL truly work **together**, a fake endpoint that only returns `{"ok": true}` is not enough—it could pass even when the database password is wrong!

Instead, our FastAPI `/health` route runs a tiny, ultra-fast query (`SELECT 1`) against PostgreSQL:
- If PostgreSQL answers: return **HTTP 200** `{"status": "ok", "database": "connected"}`.
- If PostgreSQL is unreachable: return **HTTP 503 Service Unavailable** so monitoring tools and Docker health checks immediately spot the failure.

Notice too how `DATABASE_URL` comes from `os.getenv("DATABASE_URL", ...)`. We never hardcode production passwords in Python code.

In [5]:
pyproject_text = '''[project]
name = "team-api"
version = "0.1.0"
requires-python = ">=3.13"
dependencies = [
    "fastapi>=0.115.0",
    "uvicorn[standard]>=0.32.0",
    "sqlalchemy>=2.0.36",
    "psycopg[binary]>=3.2.3",
]
'''

env_example_text = '''POSTGRES_USER=harbour
POSTGRES_PASSWORD=harbour_local_pw
POSTGRES_DB=harbour_db
'''

main_py_text = '''import os
from fastapi import FastAPI, HTTPException
from sqlalchemy import create_engine, text

# Notice the hostname is 'db' (the Docker Compose service name), NOT 'localhost'!
DATABASE_URL = os.getenv(
    "DATABASE_URL",
    "postgresql+psycopg://harbour:harbour_local_pw@db:5432/harbour_db",
)

# pool_pre_ping=True checks that a pooled connection is still alive before using it
engine = create_engine(DATABASE_URL, pool_pre_ping=True)
app = FastAPI(title="Week 4 Team API")


@app.get("/health")
def health() -> dict[str, str]:
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        return {"status": "ok", "database": "connected"}
    except Exception as exc:
        raise HTTPException(
            status_code=503,
            detail={"status": "unhealthy", "reason": "database unavailable"},
        ) from exc
'''

(stack_dir / "pyproject.toml").write_text(pyproject_text, encoding="utf-8")
# A real uv.lock must be generated before building; see the accompanying setup guide.
(stack_dir / ".env.example").write_text(env_example_text, encoding="utf-8")
(stack_dir / "app" / "__init__.py").write_text("", encoding="utf-8")
(stack_dir / "app" / "main.py").write_text(main_py_text, encoding="utf-8")

print("Wrote pyproject.toml, .env.example, and app/main.py!")
print("-" * 60)
print(main_py_text.strip())

Wrote pyproject.toml, .env.example, and app/main.py!
------------------------------------------------------------
import os
from fastapi import FastAPI, HTTPException
from sqlalchemy import create_engine, text

# Notice the hostname is 'db' (the Docker Compose service name), NOT 'localhost'!
DATABASE_URL = os.getenv(
    "DATABASE_URL",
    "postgresql+psycopg://harbour:harbour_local_pw@db:5432/harbour_db",
)

# pool_pre_ping=True checks that a pooled connection is still alive before using it
engine = create_engine(DATABASE_URL, pool_pre_ping=True)
app = FastAPI(title="Week 4 Team API")


@app.get("/health")
def health() -> dict[str, str]:
    try:
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        return {"status": "ok", "database": "connected"}
    except Exception as exc:
        raise HTTPException(
            status_code=503,
            detail={"status": "unhealthy", "reason": "database unavailable"},
        ) from exc


## 4. Docker Compose: Running API + Postgres Together

Running `docker run` by hand for multiple containers with networks, volumes, and environment variables gets messy fast. **Docker Compose** (`compose.yaml`) lets us define the entire multi-container system in one clean YAML file and start everything with a single command.

```text
   Your Browser / curl                 Docker Private Network (default)
  ┌───────────────────┐          ┌──────────────────────────────────────────┐
  │ localhost:8000    │ ───────▶ │  Service: api          Service: db       │
  └───────────────────┘  ports   │  (0.0.0.0:8000) ────▶  (db:5432)         │
                        8000:8000│         │                  │             │
                                 │   depends_on:         pg_isready         │
                                 │  service_healthy      healthcheck        │
                                 └────────────────────────────┼─────────────┘
                                                              ▼
                                                  ┌───────────────────────┐
                                                  │ Named Volume:         │
                                                  │ postgres_data         │
                                                  └───────────────────────┘
```

### Two Classic Traps Compose Solves (If Written Right!)

1. **Trap 1 — Using `localhost:5432` inside `api`:**
   Each container has its own `localhost`. If `api` connects to `localhost:5432`, it looks inside the `api` container and fails with `Connection refused`! Docker Compose runs an automatic **internal DNS server** where every service is reachable by its **service name**—so `api` connects to `@db:5432`.
2. **Trap 2 — Plain `depends_on` without a Health Check:**
   Writing `depends_on: [db]` only waits until the `db` container **starts**, *not* until PostgreSQL finishes setting up its database files and accepting SQL connections! Without a health check, `api` often boots 2 seconds too early and crashes on startup.
   - **The Fix:** Give `db` a `healthcheck` using PostgreSQL's built-in `pg_isready` tool, and set `depends_on: db: condition: service_healthy` on `api`.
> **Important:** `pg_isready` checks whether the server accepts connections; it does not prove your API credentials work. The API health endpoint tests a real database query. Startup ordering does not handle later outages; applications still need reconnect and retry logic. The sample password is for local learning only. Avoid publishing the database port in production, and use proper secret management. Special characters in passwords must be URL-encoded in a database URL.


In [6]:
# Modern Compose Specification: name the file `compose.yaml` and omit the obsolete `version:` field!
compose_yaml_text = '''services:
  db:
    image: postgres:16-alpine
    restart: unless-stopped
    environment:
      POSTGRES_USER: ${POSTGRES_USER:-harbour}
      POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-harbour_local_pw}
      POSTGRES_DB: ${POSTGRES_DB:-harbour_db}
    ports:
      - "5432:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
    healthcheck:
      # Double $$ tells Compose to pass $POSTGRES_USER into the container shell rather than expanding on the host
      test: ["CMD-SHELL", "pg_isready -U $$POSTGRES_USER -d $$POSTGRES_DB"]
      interval: 5s
      timeout: 5s
      retries: 5
      start_period: 10s

  api:
    build:
      context: .
      dockerfile: Dockerfile
    restart: unless-stopped
    environment:
      DATABASE_URL: postgresql+psycopg://${POSTGRES_USER:-harbour}:${POSTGRES_PASSWORD:-harbour_local_pw}@db:5432/${POSTGRES_DB:-harbour_db}
    ports:
      - "8000:8000"
    depends_on:
      db:
        condition: service_healthy
    healthcheck:
      # Slim Python images omit curl; Python's built-in urllib checks /health cleanly with zero extra packages
      test:
        [
          "CMD",
          "python",
          "-c",
          "import urllib.request; urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3)",
        ]
      interval: 10s
      timeout: 5s
      retries: 3
      start_period: 15s

volumes:
  postgres_data:
'''

(stack_dir / "compose.yaml").write_text(compose_yaml_text, encoding="utf-8")
print(f"Wrote {stack_dir / 'compose.yaml'}!\n")
for line_no, line in enumerate(compose_yaml_text.splitlines(), start=1):
    print(f"{line_no:02d} | {line}")

Wrote /tmp/week4_role3_stack_17keo8uw/compose.yaml!

01 | services:
02 |   db:
03 |     image: postgres:16-alpine
04 |     restart: unless-stopped
05 |     environment:
06 |       POSTGRES_USER: ${POSTGRES_USER:-harbour}
07 |       POSTGRES_PASSWORD: ${POSTGRES_PASSWORD:-harbour_local_pw}
08 |       POSTGRES_DB: ${POSTGRES_DB:-harbour_db}
09 |     ports:
10 |       - "5432:5432"
11 |     volumes:
12 |       - postgres_data:/var/lib/postgresql/data
13 |     healthcheck:
14 |       # Double $$ tells Compose to pass $POSTGRES_USER into the container shell rather than expanding on the host
15 |       test: ["CMD-SHELL", "pg_isready -U $$POSTGRES_USER -d $$POSTGRES_DB"]
16 |       interval: 5s
17 |       timeout: 5s
18 |       retries: 5
19 |       start_period: 10s
20 | 
21 |   api:
22 |     build:
23 |       context: .
24 |       dockerfile: Dockerfile
25 |     restart: unless-stopped
26 |     environment:
27 |       DATABASE_URL: postgresql+psycopg://${POSTGRES_USER:-harbour}:${POSTGRES_

In [7]:
# Let's trace the exact second-by-second startup difference for the team!
def simulate_startup(wait_for_healthy: bool) -> None:
    title = (
        "WITH Health Check (`condition: service_healthy`)"
        if wait_for_healthy
        else "WITHOUT Health Check (plain `depends_on: [db]`)"
    )
    print(f"=== {title} ===")
    db_ready_at = 4  # In this illustration Postgres needs 4 seconds on first boot to initialize data files
    api_started = False

    for sec in range(0, 7):
        db_state = "healthy (accepting SQL)" if sec >= db_ready_at else "starting (initializing files...)"
        print(f"  t={sec}s | db: {db_state}")

        can_start_api = (sec >= db_ready_at) if wait_for_healthy else (sec >= 1)
        if can_start_api and not api_started:
            api_started = True
            if sec < db_ready_at:
                print(f"       | api: starts at t={sec}s -> 💥 CRASH! Connection refused on db:5432")
                break
            else:
                print(f"       | api: starts at t={sec}s -> ✅ Connected to db:5432 cleanly on first try!")
    print()


simulate_startup(wait_for_healthy=False)
simulate_startup(wait_for_healthy=True)

=== WITHOUT Health Check (plain `depends_on: [db]`) ===
  t=0s | db: starting (initializing files...)
  t=1s | db: starting (initializing files...)
       | api: starts at t=1s -> 💥 CRASH! Connection refused on db:5432

=== WITH Health Check (`condition: service_healthy`) ===
  t=0s | db: starting (initializing files...)
  t=1s | db: starting (initializing files...)
  t=2s | db: starting (initializing files...)
  t=3s | db: starting (initializing files...)
  t=4s | db: healthy (accepting SQL)
       | api: starts at t=4s -> ✅ Connected to db:5432 cleanly on first try!
  t=5s | db: healthy (accepting SQL)
  t=6s | db: healthy (accepting SQL)



## 5. Understanding `healthcheck` Fields & Volume Lifecycle

### What the 5 `healthcheck` settings mean
- **`test`**: The command Docker runs *inside* the container. Exit code `0` = pass, non-zero = fail.
  - For Postgres, `pg_isready -U $$POSTGRES_USER -d $$POSTGRES_DB` asks the database server directly if it is ready for queries.
- **`start_period: 10s`**: Grace time while the container boots—early failures during these 10 seconds do not count against `retries`.
- **`interval: 5s`**: How often Docker runs the check.
- **`timeout: 5s`**: If the check hangs longer than 5 seconds, count it as failed.
- **`retries: 5`**: How many failures in a row mark the container `unhealthy`.

### Where does Postgres data go when you stop Compose?

| Command | Stops Containers? | Removes Containers & Network? | Deletes `postgres_data` Volume? | When to use it |
|---|---|---|---|---|
| `docker compose stop` | Yes | No | **No (Data safe)** | Pausing work for lunch |
| `docker compose down` | Yes | Yes | **No (Data safe)** | Normal end-of-day cleanup |
| `docker compose down -v` | Yes | Yes | **YES! Wipes DB clean** | Starting fresh after changing initial DB setup |

## 6. Automated Stack Audit: Catching Bugs Before `docker compose up`

Great engineers verify configuration rules automatically. Let's run a Python audit across the exact files we just generated in `stack_dir` to check 10 production readiness rules that reviewers look for in pull requests.

In [8]:
dockerfile_content = (stack_dir / "Dockerfile").read_text(encoding="utf-8")
dockerignore_content = (stack_dir / ".dockerignore").read_text(encoding="utf-8")
compose_content = (stack_dir / "compose.yaml").read_text(encoding="utf-8")

checks = [
    (
        ".dockerignore excludes .env and .venv",
        ".env" in dockerignore_content and ".venv" in dockerignore_content,
    ),
    (
        "Dockerfile copies pyproject.toml before app/ for layer caching",
        dockerfile_content.index("COPY pyproject.toml") < dockerfile_content.index("COPY --chown=appuser:appuser app"),
    ),
    (
        "Dockerfile switches to a non-root USER",
        "\nUSER appuser" in dockerfile_content,
    ),
    (
        "Dockerfile binds Uvicorn to 0.0.0.0 (not 127.0.0.1)",
        '"0.0.0.0"' in dockerfile_content and '"127.0.0.1"' not in dockerfile_content,
    ),
    (
        "compose.yaml omits the deprecated top-level `version:` key",
        not compose_content.lstrip().startswith("version:"),
    ),
    (
        "db service mounts persistent named volume `postgres_data`",
        "postgres_data:/var/lib/postgresql/data" in compose_content and "\nvolumes:\n  postgres_data:" in compose_content,
    ),
    (
        "db service defines a `pg_isready` healthcheck",
        "pg_isready" in compose_content and "healthcheck:" in compose_content,
    ),
    (
        "api DATABASE_URL connects via Compose DNS `@db:5432` (not localhost)",
        "@db:5432/" in compose_content and "@localhost:5432" not in compose_content,
    ),
    (
        "api waits for `condition: service_healthy` on db",
        "condition: service_healthy" in compose_content,
    ),
    (
        "api exposes its own `/health` check",
        "/health" in compose_content,
    ),
]

passed = 0
for idx, (label, ok) in enumerate(checks, start=1):
    badge = "PASS ✅" if ok else "FAIL ❌"
    print(f"  {idx:02d}. [{badge}] {label}")
    passed += int(ok)

assert passed == len(checks), "One or more readiness checks failed!"
print(f"\nAudit result: {passed}/{len(checks)} checks passed. Static checks passed; this is not a Docker runtime test.")

  01. [PASS ✅] .dockerignore excludes .env and .venv
  02. [PASS ✅] Dockerfile copies pyproject.toml before app/ for layer caching
  03. [PASS ✅] Dockerfile switches to a non-root USER
  04. [PASS ✅] Dockerfile binds Uvicorn to 0.0.0.0 (not 127.0.0.1)
  05. [PASS ✅] compose.yaml omits the deprecated top-level `version:` key
  06. [PASS ✅] db service mounts persistent named volume `postgres_data`
  07. [PASS ✅] db service defines a `pg_isready` healthcheck
  08. [PASS ✅] api DATABASE_URL connects via Compose DNS `@db:5432` (not localhost)
  09. [PASS ✅] api waits for `condition: service_healthy` on db
  10. [PASS ✅] api exposes its own `/health` check

Audit result: 10/10 checks passed. Static checks passed; this is not a Docker runtime test.


## 7. Verifying It Runs: The 5-Step Verification Playbook

Part of Member 3's role is not only writing the files, but **proving that both containers start, pass their health checks, and talk to each other**. The generated example still needs a **real uv.lock** before building (see the accompanying setup guide). The audit above checks text only; it does not prove containers run. Here is the verification flow:

```bash
# 1. Build the API image and start both services in the background (-d = detached)
docker compose up -d --build

# 2. Check container state — look for `(healthy)` under STATUS for BOTH db and api!
docker compose ps

# 3. Call the API health endpoint from your host terminal (proves Host -> API -> Postgres -> Host)
curl -i http://localhost:8000/health

# 4. Inspect recent logs if anything looks stuck or unhealthy
docker compose logs --tail=30 db api

# 5. Stop containers cleanly while keeping your Postgres data volume intact
docker compose down
```

### What `docker compose ps` looks like when everything is right
Notice `(healthy)` right next to `Up`:
```text
NAME          IMAGE                STATUS                   PORTS
project-db-1  postgres:16-alpine   Up 24 seconds (healthy)  0.0.0.0:5432->5432/tcp
project-api-1 project-api          Up 18 seconds (healthy)  0.0.0.0:8000->8000/tcp
```

In [8]:
import shutil
import subprocess

print("Generated reference stack tree:")
for path in sorted(stack_dir.rglob("*")):
    if path.is_file():
        rel = path.relative_to(stack_dir)
        print(f"  📄 {rel}")

print("\nChecking if Docker CLI is present in this notebook environment...")
docker_bin = shutil.which("docker")
if docker_bin:
    info = subprocess.run(["docker", "--version"], capture_output=True, text=True, check=False)
    print("  Docker CLI detected:", info.stdout.strip() or info.stderr.strip())
else:
    print("  Docker CLI is not running inside this notebook kernel (expected in sandboxes).")

expected_health_json = '{"status":"ok","database":"connected"}'
print(f"\nExpected HTTP 200 response from `curl -s http://localhost:8000/health`:")
print(f"  {expected_health_json}")

Generated reference stack tree:
  📄 .dockerignore
  📄 .env.example
  📄 Dockerfile
  📄 app/__init__.py
  📄 app/main.py
  📄 compose.yaml
  📄 pyproject.toml

Checking if Docker CLI is present in this notebook environment...
  Docker CLI detected: Docker version 29.9.3, build 5457199

Expected HTTP 200 response from `curl -s http://localhost:8000/health`:
  {"status":"ok","database":"connected"}


## 8. Team Huddle: 5 Real-World Debugging Scenarios

Try answering each scenario with the team before clicking to reveal the answer!

<details>
<summary><strong>1. A teammate changes Uvicorn's host to <code>127.0.0.1</code> inside the Dockerfile. The container says "Uvicorn running", yet <code>http://localhost:8000/health</code> fails in their browser. Why?</strong></summary>
<br>
Inside a container, <code>127.0.0.1</code> only listens on the container's own internal loopback interface. Traffic mapped from your laptop via <code>ports: ["8000:8000"]</code> arrives on the container's virtual network card, so Uvicorn must bind to <code>0.0.0.0</code>.
</details>

<details>
<summary><strong>2. In <code>.env</code>, someone sets <code>DATABASE_URL=postgresql+psycopg://harbour:pw@localhost:5432/harbour_db</code> for the <code>api</code> container. What happens?</strong></summary>
<br>
The <code>api</code> container tries to connect to port <code>5432</code> inside <em>itself</em> instead of the Postgres container, failing with <code>Connection refused</code>. Inside Compose, use the service name: <code>@db:5432</code>.
</details>

<details>
<summary><strong>3. Every time someone changes a comment in <code>app/main.py</code>, <code>docker compose build</code> spends 30 seconds reinstalling all Python packages. What went wrong in the Dockerfile?</strong></summary>
<br>
<code>COPY . .</code> was placed <em>above</em> <code>RUN uv sync</code>, breaking Docker's layer cache whenever any source file changed. Copy <code>pyproject.toml</code> and <code>uv.lock</code> first, run <code>uv sync</code>, and copy <code>app/</code> after.
</details>

<details>
<summary><strong>4. You insert 50 test rows into Postgres, run <code>docker compose down</code>, and then run <code>docker compose up -d</code>. Are the 50 rows still there?</strong></summary>
<br>
<strong>Yes!</strong> Because <code>postgres_data</code> is a named volume mounted at <code>/var/lib/postgresql/data</code>, <code>docker compose down</code> removes the containers and network while keeping the volume intact. Only <code>docker compose down -v</code> deletes the volume.
</details>

<details>
<summary><strong>5. Why did we write <code>$$POSTGRES_USER</code> with two dollar signs inside <code>compose.yaml</code>'s health check?</strong></summary>
<br>
With a single <code>$</code>, Docker Compose tries to substitute the variable on the <em>host</em> before starting the container. Writing <code>$$</code> escapes it so the container's own shell reads <code>$POSTGRES_USER</code> from the container's environment!
</details>

## 9. Role 3 One-Minute Cheat Sheet

- **Image vs. Container:** An **image** is a frozen, read-only stack of layers (the recipe); a **container** is a live, isolated process with a temporary top layer running from that image (the dish).
- **Fast, Safe `Dockerfile`:** Exclude `.env` and `.venv` in `.dockerignore` → copy `pyproject.toml` + `uv.lock` before `app/` → run as a non-root user → bind Uvicorn to `0.0.0.0:8000`.
- **Reliable `compose.yaml`:** Connect `api` to Postgres via its service name (`@db:5432`), store DB files in a named volume (`postgres_data`), check Postgres readiness with `pg_isready`, and gate `api` startup with `depends_on` + `condition: service_healthy`.
- **Verify Everything:** `docker compose up -d --build` → confirm `(healthy)` in `docker compose ps` → verify HTTP `200` from `http://localhost:8000/health`.

**Next up for the team:** Open the companion **15-minute, 20-question HTML quiz** to test your mastery!